# Lab 16-05 · Windows, Noise Floors and an ADC Test

Five parts, each built on `spectral.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import spectral as sp
NAMES = ["rect", "hann", "hamming", "blackman", "bh4", "flattop"]

## Part 1 · Window spectra and figures of merit

In [2]:
N, P = 64, 64 * 64
fig, ax = plt.subplots(figsize=(8, 3.5))
nu = np.arange(P) * N / P
for name in NAMES:
    W = np.abs(np.fft.fft(sp.window(name, N), P))
    ax.plot(nu[:P // 4], 20 * np.log10(np.maximum(W[:P // 4] / W[0], 1e-8)), label=name)
ax.set_ylim(-130, 5); ax.set_xlabel("bins from the tone"); ax.set_ylabel("dB"); ax.legend(fontsize=7, ncol=3); plt.show()
print(f"{'window':9s} {'CG':>7s} {'ENBW':>7s} {'scallop':>8s} {'sidelobe':>9s}")
for name in NAMES:
    w = sp.window(name, 4096)
    print(f"{name:9s} {sp.coherent_gain(w):7.4f} {sp.enbw_bins(w):7.3f} {sp.scalloping_db(w):8.2f} {sp.highest_sidelobe_db(w):9.1f}")

window         CG    ENBW  scallop  sidelobe


rect       1.0000   1.000    -3.92     -13.3


hann       0.5000   1.500    -1.42     -31.5


hamming    0.5400   1.363    -1.75     -42.7


blackman   0.4200   1.727    -1.10     -58.1


bh4        0.3588   2.004    -0.83     -92.0


flattop    0.2156   3.770    -0.01     -93.0


## Part 2 · A weak tone beside a strong one

In [3]:
Nn = 1024
n = np.arange(Nn)
strong = np.cos(2 * np.pi * 100.5 * n / Nn)
for name in ("rect", "hann", "bh4"):
    w = sp.window(name, Nn)
    S = np.abs(np.fft.fft(strong * w)); pk = S.max()
    for level_db in range(-20, -121, -10):
        weak = 10 ** (level_db / 20) * np.cos(2 * np.pi * 110 * n / Nn)
        tot = np.abs(np.fft.fft((strong + weak) * w))
        if tot[110] < 2 * S[110]:          # the weak tone no longer doubles the bin: lost in the leakage
            print(f"{name:5s}: weak tone lost at about {level_db} dB")
            break
    else:
        print(f"{name:5s}: still visible at -120 dB")

rect : weak tone lost at about -30 dB
hann : weak tone lost at about -70 dB
bh4  : weak tone lost at about -100 dB


## Part 3 · Processing gain

*Predict:* how do the floor depth and the SNR change as $N$ doubles?

In [4]:
q = 2 / 2**12
rows = []
for p in range(10, 17):
    N = 2**p
    M = sp.coherent_bin(N / 8.2, N, N)          # any odd bin near N/8
    x = (1 - 1e-9) * np.sin(2 * np.pi * M * np.arange(N) / N + 0.3)
    xq = np.clip(np.round(x / q) * q, -1, 1 - q)
    P = np.abs(np.fft.fft(xq)) ** 2
    floor = 10 * math.log10(P[M] / np.mean(np.delete(P[1:N // 2], [M - 1])))
    rows.append((N, floor, sp.adc_metrics(xq, M)["snr"]))
for N, fl, snr in rows:
    print(f"N = {N:6d}: floor {fl:6.1f} dB below the tone, SNR {snr:5.2f} dB")

N =   1024: floor  101.2 dB below the tone, SNR 74.13 dB
N =   2048: floor  104.1 dB below the tone, SNR 74.07 dB
N =   4096: floor  107.1 dB below the tone, SNR 73.97 dB
N =   8192: floor  110.0 dB below the tone, SNR 73.87 dB
N =  16384: floor  112.9 dB below the tone, SNR 73.80 dB
N =  32768: floor  115.9 dB below the tone, SNR 73.81 dB
N =  65536: floor  119.0 dB below the tone, SNR 73.81 dB


## Part 4 · Averaging a noise spectrum

In [5]:
noise = np.random.default_rng(57).normal(size=1024 * 64)
for K in (1, 4, 16, 64):
    est, nseg = sp.welch(noise[:1024 * K], 1024, overlap=0.0, name="hann")
    band = est[10:500]
    print(f"K = {nseg:2d}: scatter/mean = {band.std() / band.mean():.3f}   (1/sqrt(K) = {1 / math.sqrt(nseg):.3f})")

K =  1: scatter/mean = 1.063   (1/sqrt(K) = 1.000)
K =  4: scatter/mean = 0.493   (1/sqrt(K) = 0.500)
K = 16: scatter/mean = 0.258   (1/sqrt(K) = 0.250)
K = 64: scatter/mean = 0.129   (1/sqrt(K) = 0.125)


## Part 5 · An ADC test, coherent and not

In [6]:
fs, N = 100e6, 16384
M = sp.coherent_bin(10e6, fs, N)
A = 10 ** (-1 / 20); a3 = 4e-4 / A**2; a2 = 2 * 10 ** (-90 / 20) / A; q = 2 / 2**12


def adc(cycles, seed=1605):
    rg = np.random.default_rng(seed)
    nn = np.arange(N)
    x = A * np.sin(2 * np.pi * cycles * nn / N + 0.7)
    y = x + a2 * x**2 + a3 * x**3 + rg.normal(scale=2.0e-4, size=N)
    return np.clip(np.round(y / q) * q, -1, 1 - q)


yc = adc(M)
m = sp.adc_metrics(yc, M)
amp, res = sp.sine_fit3(yc, M / N)
print(f"coherent (M = {M}): SNR {m['snr']:.2f}, SINAD {m['sinad']:.2f}, THD {m['thd']:.2f}, SFDR {m['sfdr']:.2f} dB, ENOB {m['enob']:.2f}")
print(f"sine-fit SINAD {20 * math.log10(amp / math.sqrt(2) / res):.2f} dB")
yn = adc(M + 0.3)
print(f"non-coherent, naive rectangular: SINAD {sp.adc_metrics(yn, M)['sinad']:.2f} dB")
for name in ("rect", "hann", "bh4"):
    r = sp.adc_metrics_window(yn, name, M + 0.3)
    print(f"non-coherent, {name:4s} with guard bands: SINAD {r['sinad']:.2f} dB, ENOB {r['enob']:.2f}")

coherent (M = 1637): SNR 68.15, SINAD 67.85, THD -79.60, SFDR 79.97 dB, ENOB 10.98
sine-fit SINAD 67.85 dB
non-coherent, naive rectangular: SINAD 4.47 dB
non-coherent, rect with guard bands: SINAD 14.02 dB, ENOB 2.04
non-coherent, hann with guard bands: SINAD 49.77 dB, ENOB 7.97
non-coherent, bh4  with guard bands: SINAD 67.82 dB, ENOB 10.97


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.